# Top-k vs. Brute Force

Compare a format choice based on selected multiplication terms with choices based on the full output.


## 0 · Imports

In [1]:
import math
import time
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

try:
    from IPython.display import display
except ImportError:
    display = print

def show(df):
    """Plain table display with generous row/column limits."""

    with pd.option_context("display.max_rows", 200, "display.max_columns", None,
                           "display.precision", 6, "display.width", 200):
        display(df)

from scipy.stats import kendalltau


## Settings

Set the weight and input file paths. Choose what percentage of large entries to keep. `union` uses selections from both matrices; `w_only` or `x_only` uses one. The remaining settings control storage limits and the size of the scale search.


In [2]:
W_PATH = "c:/Users/Admin/OneDrive/Desktop/System/W_C.npy"
X_CAL_PATH = "c:/Users/Admin/OneDrive/Desktop/System/X_cal.npy"
DATASET_NAME = "C"

K_W_PCT = 20.0
K_X_PCT = 20.0
SELECTION_MODE = "union"
MAX_BITS_PER_WEIGHT = None

REF_WINDOW = 3
REF_WINDOW_MAX = 5
REF_MAX_ELEMS = 2_000_000

EXPORT_RESULTS = True
OUTPUT_DIR = (Path("/kaggle/working/topk_vs_bruteforce_results") if Path("/kaggle/working").is_dir()
              else Path.cwd() / "topk_vs_bruteforce_results")


## Prepare the number formats

The next two cells define five 4-bit formats and how they round weights. Groups of nearby weights share a scale. Changing the scale changes the step size between allowed values.


In [3]:
ASSIGNED_ELEMENT_BITS = 4

E8M0_MIN_EXP = - 127
E8M0_MAX_EXP = 127

BFP_EXP_POLICY = "autoflex_predict"
BFP_ALPHA = 1.0
BFP_SHARING = "block"
BFP_BLOCK_SIZE = 32

FORMAT_CONFIGS = {
    "E2M1": {
        "kind": "mxfp",
        "block_size": 32,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "[S|E|M] = [1|2|1]",
        "grid": np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0], dtype = np.float64),
        "baseline_exp_shift": - 2,
        "exp_min": E8M0_MIN_EXP,
        "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; 8-bit E8M0-style shared exponent",
        "exceptions": "finite values only; no NaN/Inf codes are emitted",
        "convention_status": "classroom/OCP-MX convention used in this project",
    },
    "E1M2": {
        "kind": "mxfp",
        "block_size": 32,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "[S|E|M] = [1|1|2]",
        "grid": np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5], dtype = np.float64),
        "baseline_exp_shift": - 1,
        "exp_min": E8M0_MIN_EXP,
        "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; 8-bit E8M0-style shared exponent",
        "exceptions": "finite values only; no NaN/Inf codes are emitted",
        "convention_status": "adopted classroom-consistent E1M2 convention",
    },
    "MXINT4": {
        "kind": "mxint",
        "block_size": 32,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "[S|I|F] = [1|1|2]",
        "qmin": - 8,
        "qmax": 7,
        "element_scale": 0.25,
        "exp_min": E8M0_MIN_EXP,
        "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; element precision scale 2^-2",
        "exceptions": "signed integer codes -8..7; no NaN/Inf codes",
        "convention_status": "adopted from the classroom MXINT construction",
    },
    "BFP4": {
        "kind": "bfp",
        "block_size": BFP_BLOCK_SIZE,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "flex4+8: 4-bit two's-complement integer mantissa (codes -8..7) + shared exponent",
        "mantissa_bits": 4,
        "qmin": - 8,
        "qmax": 7,
        "exp_min": E8M0_MIN_EXP,
        "exp_max": E8M0_MAX_EXP,
        "shared_scale": "kappa = 2^E, value = m * kappa; 8-bit shared exponent per group",
        "exceptions": "finite two's-complement integers only; saturate to [-8,7]; no NaN/Inf codes",
        "convention_status": "Flexpoint (Koster et al., arXiv:1711.02213), Autoflex exponent rule",
    },
    "MXUINT4": {
        "kind": "mxuint",
        "block_size": 32,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "Unsigned 4-bit code; fixed midpoint zero point q_u - 8",
        "shared_scale": "power-of-two 2^e; fixed zero point 8; no per-block offset",
        "exceptions": "finite values; unsigned codes 0..15 decode to signed integers -8..7",
        "convention_status": "custom midpoint-coded unsigned-storage convention; no FP32 offset overhead",
    },
}

SCORED_FORMATS = ["E2M1", "E1M2", "MXUINT4", "MXINT4", "BFP4"]

assert all(FORMAT_CONFIGS[f]["element_bits"] == ASSIGNED_ELEMENT_BITS for f in SCORED_FORMATS)


In [4]:
def rowwise_ranges(cols, block_size):

    for start in range(0, cols, block_size):
        yield start, min(start + block_size, cols)

def storage_for_format(fmt, shape):
    """Theoretical packed storage: element bits + one metadata word per block."""
    cfg = FORMAT_CONFIGS[fmt]
    n = shape[0] * shape[1]
    if fmt == "BFP4" and BFP_SHARING == "tensor":
        n_blocks = 1
    else:
        n_blocks = shape[0] * math.ceil(shape[1] / cfg["block_size"])

    element_bits = cfg["element_bits"] * n
    metadata_bits = cfg["metadata_bits"] * n_blocks
    total = element_bits + metadata_bits
    return {"Blocks": n_blocks, "Element_Bits": element_bits, "Metadata_Bits": metadata_bits,
            "Total_Storage_Bits": total, "Effective_Bits_per_Weight": total / n}

def nearest_grid_rne(values, grid):
    """Choose the nearest grid value; ties prefer an even code index."""
    v = np.asarray(values, dtype = np.float64)
    flat = v.reshape( - 1)
    d = np.abs(flat[:, None] - grid[None, :])
    dmin = d.min(axis = 1, keepdims = True)
    tie = np.isclose(d, dmin, rtol = 0.0, atol = 1e-15)
    even = tie & ((np.arange(grid.size) % 2) == 0)[None, :]
    pick = np.where(even.any(axis = 1, keepdims = True), even, tie)
    return grid[pick.argmax(axis = 1)].reshape(v.shape)

def _bfp_gamma(block, E):
    """Gamma of the Flexpoint paper: max |mantissa| at kappa = 2^E (saturated)."""
    N = FORMAT_CONFIGS["BFP4"]["mantissa_bits"]
    m = np.rint(np.abs(np.asarray(block, dtype = np.float64)) / np.exp2(float(E)))
    return float(min(np.max(m), 2 ** (N - 1) - 1))

def bfp_exp_autoflex_predict(block):
    """Paper Algorithm 2, line 10 for a static tensor:  E = ceil(log2 chi) - N + 1."""
    N = FORMAT_CONFIGS["BFP4"]["mantissa_bits"]
    chi = BFP_ALPHA * float(np.max(np.abs(block)))
    if chi == 0:
        return 0

    return int(np.ceil(np.log2(chi))) - N + 1

def bfp_exp_autoflex_init(block):
    """Paper Algorithm 1 (trial-and-error initialisation), implemented literally."""
    N = FORMAT_CONFIGS["BFP4"]["mantissa_bits"]
    if float(np.max(np.abs(block))) == 0:
        return 0

    E = 0

    for _ in range(1024):
        G = _bfp_gamma(block, E)
        if G >= 2 ** (N - 1) - 1:
            E += (N - 1) // 2
        elif G < 2 ** (N - 2):
            E += int(np.ceil(np.log2(max(G, 1.0)))) - (N - 2)
            if G > 2.0 ** ((N - 1) // 2 - 2):
                break
        else:
            break

        E = int(np.clip(E, E8M0_MIN_EXP, E8M0_MAX_EXP))

    return E

def bfp_shared_exponent(block):
    if BFP_EXP_POLICY == "autoflex_init":
        return bfp_exp_autoflex_init(block)

    if BFP_EXP_POLICY == "autoflex_predict":
        return bfp_exp_autoflex_predict(block)

    raise ValueError(f"Unknown BFP_EXP_POLICY={BFP_EXP_POLICY}")

def baseline_exp(block, fmt):
    """Baseline shared exponent of one block (no offset search)."""
    amax = float(np.max(np.abs(block)))
    if amax == 0:
        return 0

    raw = int(np.floor(np.log2(amax)))
    cfg = FORMAT_CONFIGS[fmt]
    if cfg["kind"] == "mxfp":
        exp = raw + cfg["baseline_exp_shift"]
    elif cfg["kind"] == "mxuint":

        neg_extent = max(0.0, - float(np.min(block)))
        pos_extent = max(0.0, float(np.max(block)))
        required_step = max(neg_extent / 8.0, pos_extent / 7.0)
        exp = int(np.ceil(np.log2(required_step))) if required_step > 0 else 0
    elif cfg["kind"] == "bfp":
        exp = bfp_shared_exponent(block)
    else:
        exp = raw

    if "exp_min" in cfg:
        exp = int(np.clip(exp, cfg["exp_min"], cfg["exp_max"]))

    return exp

def quantize_block_at_exp(block, fmt, exp):
    """De-quantised values of one block at shared exponent `exp`."""
    block = np.asarray(block, dtype = np.float64)
    cfg = FORMAT_CONFIGS[fmt]
    if "exp_min" in cfg:
        exp = int(np.clip(exp, cfg["exp_min"], cfg["exp_max"]))

    if cfg["kind"] == "mxfp":
        scale = np.exp2(exp)
        qlocal = nearest_grid_rne(np.abs(block) / scale, cfg["grid"])
        return np.sign(block) * qlocal * scale

    if cfg["kind"] == "mxint":
        step = np.exp2(exp) * cfg["element_scale"]
        return np.clip(np.rint(block / step), cfg["qmin"], cfg["qmax"]) * step

    if cfg["kind"] == "bfp":
        scale = np.exp2(exp)
        return np.clip(np.rint(block / scale), cfg["qmin"], cfg["qmax"]) * scale

    if cfg["kind"] == "mxuint":
        scale = np.exp2(exp)
        q = np.clip(np.rint(block / scale) + 8, 0, 15)
        return (q - 8) * scale

    raise ValueError(f"Numerical quantizer not defined for {fmt}")

def quantize_matrix_baseline(W, fmt):
    """Quantise the whole matrix with baseline exponents. Returns (Wq, exponents)."""
    cfg = FORMAT_CONFIGS[fmt]
    if fmt == "BFP4" and BFP_SHARING == "tensor":
        E = baseline_exp(W.ravel(), fmt)
        return quantize_block_at_exp(W, fmt, E), np.array([[E]], dtype = np.int16)

    ranges = list(rowwise_ranges(W.shape[1], cfg["block_size"]))
    Wq = np.zeros_like(W)
    exps = np.zeros((W.shape[0], len(ranges)), dtype = np.int16)

    for r in range(W.shape[0]):

        for b, (s, e) in enumerate(ranges):
            exps[r, b] = baseline_exp(W[r, s:e], fmt)
            Wq[r, s:e] = quantize_block_at_exp(W[r, s:e], fmt, exps[r, b])

    return Wq, exps

def relative_error_pct(sse, energy):
    if energy == 0:
        return 0.0 if sse == 0 else float("inf")

    return 100.0 * math.sqrt(max(0.0, sse) / energy)


## 3 · Data loading

In [5]:
def validated_arrays(W_input, X_input):

    for name, a in [("W", W_input), ("X_cal", X_input)]:
        if not isinstance(a, np.ndarray) or a.ndim != 2 or min(a.shape) == 0:
            raise ValueError(f"{name} must be a nonempty 2-D numeric array.")

        if not np.issubdtype(a.dtype, np.number) or np.iscomplexobj(a):
            raise ValueError(f"{name} must contain real numeric values.")

        if not np.isfinite(a).all():
            raise ValueError(f"{name} contains NaN/Inf.")

    if W_input.shape[1] != X_input.shape[0]:
        raise ValueError(f"Need W.shape[1] == X_cal.shape[0], got {W_input.shape}, {X_input.shape}.")

    return np.asarray(W_input, dtype = np.float64), np.asarray(X_input, dtype = np.float64)

def load_npy(path):
    p = Path(path).expanduser()
    if not p.is_file():
        raise FileNotFoundError(f"File not found: {p}\nSet the exact path in the configuration cell.")

    return np.load(p, allow_pickle = False)

def load_cases(w_path, x_cal_path, name):
    """Load one weight matrix W and the calibration matrix X_cal from their full paths."""
    W, X = validated_arrays(load_npy(w_path), load_npy(x_cal_path))
    return {str(name): dict(W = W, X = X, W_source = str(w_path), X_source = str(x_cal_path))}


## Select multiplication terms and score formats

This method selects individual entries in the weights and inputs. It does not select entire columns or rows.

1. **Mark large entries.** Find the largest absolute entries in `W` and `X_cal`, using the configured percentages. Absolute value makes large negative entries important too.
2. **Keep multiplication terms.** In `union` mode, keep a term when its weight entry or its input entry is marked. A term marked by both is counted once. `w_only` and `x_only` use just one set of marks.
3. **Test each format.** Quantize the full weight matrix at baseline block scales. Use the selected multiplication terms to calculate output error contributions.
4. **Choose a winner.** Add the selected signed error contributions first, then square and sum the output errors. The smallest score wins; exact ties use storage, then format order.

**Example:** one output is `w1*x1 + w2*x2 + w3*x3`. If `w2` and `x3` are marked, union mode keeps terms 2 and 3. If `x2` is marked as well, term 2 still appears only once.

The score uses selected terms, so it may rank formats differently from the full output. Also, the selector still quantizes the full weight matrix: the reduction is in terms used for scoring.


In [6]:
def top_mask(a, pct):
    """Boolean mask of the largest `pct` % entries by |value|. Ties -> original flattened order."""
    if not np.isfinite(pct) or not 0 <= pct <= 100:
        raise ValueError("Percentages must be between 0 and 100.")

    k = int(math.ceil(a.size * pct / 100.0))
    order = np.argsort( - np.abs(a).ravel(), kind = "stable")[:k]
    mask = np.zeros(a.size, dtype = bool)
    mask[order] = True
    return mask.reshape(a.shape)

def build_plan(W, X, kw, kx, mode):
    """Masks + duplicate-free bookkeeping of the union of selected multiplication terms (r, i, j)."""
    if mode not in ("union", "w_only", "x_only"):
        raise ValueError("Invalid SELECTION_MODE.")

    mw, mx = top_mask(W, kw), top_mask(X, kx)
    if mode == "w_only":
        mx[:] = False

    if mode == "x_only":
        mw[:] = False

    R, I = W.shape
    J = X.shape[1]
    w_counts = mw.sum(axis = 0, dtype = np.int64)
    x_counts = mx.sum(axis = 1, dtype = np.int64)
    overlap = int(w_counts @ x_counts)
    total = int(R * I * J)
    terms = int(mw.sum()) * J + int(mx.sum()) * R - overlap
    if terms == 0:
        raise ValueError("Selection is empty. Increase the active percentage.")

    xs = csr_matrix(X * mx)
    xs.eliminate_zeros()
    return dict(mw = mw, mx = mx, X = X, Xs = xs, terms = terms, total = total, overlap = overlap, mode = mode)

def selected_product(V, plan):
    """T(V) = (V*Mw) X + (V*(1-Mw)) (X*Mx): every selected term exactly once, no 3-D mask."""
    vs = csr_matrix(V * plan["mw"])
    vs.eliminate_zeros()
    first = np.asarray(vs @ plan["X"])
    rest = V * (~plan["mw"])
    second = np.asarray(plan["Xs"].T @ rest.T).T
    return first + second

def eligibility_table(shape):
    rows = []

    for fmt in SCORED_FORMATS:
        st = storage_for_format(fmt, shape)
        status, reason = "ELIGIBLE", "Encoding is defined in the configuration."
        if MAX_BITS_PER_WEIGHT is not None and st["Effective_Bits_per_Weight"] > MAX_BITS_PER_WEIGHT + 1e-12:
            status, reason = "OVER_STORAGE_LIMIT", "Effective storage exceeds the supplied limit."

        rows.append(dict(Format = fmt, Status = status, Reason = reason, Blocks = st["Blocks"],
                         Total_Storage_Bits = st["Total_Storage_Bits"],
                         Bits_per_Weight = st["Effective_Bits_per_Weight"]))

    return pd.DataFrame(rows)

def baseline_quantize(W, fmt):
    return quantize_matrix_baseline(W, fmt)

def select_case(W, X):
    """Rank formats by selected-term error at baseline exponents."""
    start = time.perf_counter()
    plan = build_plan(W, X, K_W_PCT, K_X_PCT, SELECTION_MODE)
    target = selected_product(W, plan)
    selected_energy = float(np.sum(target ** 2))
    elig = eligibility_table(W.shape)
    eligible = elig.loc[elig.Status == "ELIGIBLE", "Format"].tolist()
    if not eligible:
        raise ValueError("No eligible format satisfies the configured encoding/storage constraints.")

    if selected_energy == 0:
        raise ValueError("Selected target has zero energy. Increase percentages or inspect cancellation.")

    setup_seconds = time.perf_counter() - start
    scores, row_records = [], []
    best = None
    best_q = best_exponents = None

    for order, fmt in enumerate(eligible):
        t = time.perf_counter()
        Q, exponents = baseline_quantize(W, fmt)
        quant_seconds = time.perf_counter() - t
        t = time.perf_counter()
        residual = selected_product(W - Q, plan)
        loss_rows = np.sum(residual ** 2, axis = 1)
        sse = float(loss_rows.sum())
        score_seconds = time.perf_counter() - t
        bits = storage_for_format(fmt, W.shape)["Effective_Bits_per_Weight"]
        scores.append(dict(Format = fmt, Selected_SSE = sse,
                           Selected_relative_error_pct = relative_error_pct(sse, selected_energy),
                           Quantize_seconds = quant_seconds, Score_seconds = score_seconds,
                           Bits_per_weight = bits))

        for r, loss in enumerate(loss_rows):
            row_records.append(dict(Format = fmt, Row = r, Selected_output_SSE = float(loss),
                                    Selected_target_energy = float(np.sum(target[r] ** 2))))

        key = (sse, bits, order)
        if best is None or key < best:
            best, best_q, best_exponents = key, Q.copy(), exponents.copy()
            winner = fmt

    table = pd.DataFrame(scores).sort_values(["Selected_SSE", "Bits_per_weight"], kind = "stable").reset_index(drop = True)
    table.insert(0, "Predicted_rank", np.arange(1, len(table) + 1))
    return dict(plan = plan, target = target, selected_energy = selected_energy, eligibility = elig,
                scores = table, row_losses = pd.DataFrame(row_records), winner = winner,
                Wq = best_q, exponents = best_exponents, setup_seconds = setup_seconds,
                selection_seconds = time.perf_counter() - start)

select_format = select_case

def retained_product_energy_pct(W, X, plan):
    """Share of sum_{r,i,j} (W_ri X_ij)^2 covered by the selected terms (no full product needed)."""
    w2, x2 = W * W, X * X
    total = float(w2.sum(axis = 0) @ x2.sum(axis = 1))
    omitted = float((w2 * (~plan["mw"])).sum(axis = 0) @ (x2 * (~plan["mx"])).sum(axis = 1))
    return 100.0 * (1.0 - omitted / total) if total else float("nan")


## Evaluate the full output and search for better scales

**Full baseline:** quantize every weight using the baseline block scales, multiply by the full input matrix, and compare with `W @ X_cal`. This checks whether the top-k ranking agrees with a complete evaluation of the same quantizers.

**Brute force:** allow each weight block to use nearby exponent choices. For each row and format, test combinations of these choices and keep the smallest complete output error. For example, two blocks with three choices each give nine combinations. Testing combinations matters because block errors can cancel or add together in the output.

The search usually starts within ±3 of baseline exponents and may widen up to ±5 if the best choice is on the boundary. It is exhaustive within the candidate window, not across every possible exponent. Exceeding the exact-search size limit raises an error.

A Gram factor is used to calculate the same output-error objective more efficiently. You do not need to follow that linear-algebra detail to understand the comparison: the target remains the original full multiplication.

Comparing top-k with the full baseline checks the effect of partial scoring. Comparing it with brute force also includes the effect of changing block scales.


In [7]:
def full_baseline_eval(W, X):
    """Actual output error 100*||(W-Wq)X||_F/||WX||_F for every format (baseline exponents)."""
    t0 = time.perf_counter()
    energy = float(np.sum((W @ X) ** 2))
    errors = {}

    for fmt in SCORED_FORMATS:
        Q, _ = quantize_matrix_baseline(W, fmt)
        sse = float(np.sum(((W - Q) @ X) ** 2))
        errors[fmt] = relative_error_pct(sse, energy)

    return errors, time.perf_counter() - t0

def _gram_factor(X):
    """F with F F^T = X X^T, so ||D X||_F^2 = ||D F||_F^2 (exact; F has at most rank(X) columns)."""
    G = X @ X.T
    lam, V = np.linalg.eigh(G)
    if lam.size == 0 or np.max(lam) <= 0:
        return np.zeros((X.shape[0], 0), dtype = np.float64)

    keep = lam > np.max(lam) * 1e-14
    return V[:, keep] * np.sqrt(lam[keep])

def _enumerate_exact(D):
    """Exhaustive search over all exponent combinations of one row. D[b] = per-candidate residual rows of block b."""
    S = D[0]

    for d in D[1:]:
        S = (S[:, None, :] + d[None, :, :]).reshape( - 1, S.shape[ - 1])

    sse = np.einsum("ij,ij->i", S, S)
    k = int(np.argmin(sse))
    idx = [int(i) for i in np.unravel_index(k, [len(d) for d in D])]
    return idx, float(sse[k])

def _bfp_tensor_search(W, X):
    cfg = FORMAT_CONFIGS["BFP4"]
    base = baseline_exp(W.ravel(), "BFP4")
    window = REF_WINDOW

    while True:
        exps = sorted({int(np.clip(base + o, cfg["exp_min"], cfg["exp_max"]))
                       for o in range( - window, window + 1)}, key = lambda e: (abs(e - base), e - base))
        best = None

        for E in exps:
            Wq = quantize_block_at_exp(W, "BFP4", E)
            s = float(np.sum(((W - Wq) @ X) ** 2))
            if best is None or s < best[0]:
                best = (s, E, Wq)

        if abs(best[1] - base) == window and window < REF_WINDOW_MAX:
            window += 1
            continue

        return best[2]

def exact_scale_search(W, X, fmt):
    """Brute force: for every row, enumerate ALL shared-exponent combinations of its blocks inside a window
    around the baseline exponent (+-REF_WINDOW, widened up to +-REF_WINDOW_MAX if the optimum touches the edge)
    and keep the combination with the smallest ACTUAL calibration-output error. Raises if enumeration is too big."""
    if fmt == "BFP4" and BFP_SHARING == "tensor":
        return _bfp_tensor_search(W, X)

    cfg = FORMAT_CONFIGS[fmt]
    F = _gram_factor(X)
    ranges = list(rowwise_ranges(W.shape[1], cfg["block_size"]))
    limit = max(1, REF_MAX_ELEMS // max(1, F.shape[1]))
    Wq = np.zeros_like(W)
    cache = {}

    def candidates(r, b, s, e, window):
        block = W[r, s:e]
        base = baseline_exp(block, fmt)
        exps = []

        for off in sorted(range( - window, window + 1), key = lambda o: (abs(o), o)):
            ex = base + off
            if "exp_min" in cfg:
                ex = int(np.clip(ex, cfg["exp_min"], cfg["exp_max"]))

            if ex not in exps:
                exps.append(ex)

        Q = []

        for ex in exps:
            key = (r, b, ex)
            if key not in cache:
                cache[key] = quantize_block_at_exp(block, fmt, ex)

            Q.append(cache[key])

        return base, np.asarray(exps, dtype = int), np.asarray(Q, dtype = np.float64)

    for r in range(W.shape[0]):
        T = [W[r, s:e] @ F[s:e] for s, e in ranges]
        window = REF_WINDOW

        while True:
            C = [candidates(r, b, s, e, window) for b, (s, e) in enumerate(ranges)]
            D = [T[b] - C[b][2] @ F[s:e] for b, (s, e) in enumerate(ranges)]
            combos = math.prod(len(d) for d in D)
            if combos > limit:
                raise RuntimeError(f"{fmt} row {r}: {combos:,} combinations exceed the exact-search cap {limit:,}. "
                                   "Increase REF_MAX_ELEMS or reduce REF_WINDOW; this notebook never silently "
                                   "falls back to a heuristic.")

            idx, _ = _enumerate_exact(D)
            offsets = [int(C[b][1][idx[b]] - C[b][0]) for b in range(len(ranges))]
            edge = any(abs(o) == window for o in offsets)
            can_widen = (2 * (window + 1) + 1) ** len(ranges) <= limit
            if edge and window < REF_WINDOW_MAX and can_widen:
                window += 1
                continue

            break

        for b, (s, e) in enumerate(ranges):
            Wq[r, s:e] = C[b][2][idx[b]]

    return Wq

def brute_force_reference(W, X):
    """Scale-optimised actual output error (%) for every format, plus search time."""
    t0 = time.perf_counter()
    energy = float(np.sum((W @ X) ** 2))
    errors = {}

    for fmt in SCORED_FORMATS:
        Wq = exact_scale_search(W, X, fmt)
        errors[fmt] = relative_error_pct(float(np.sum(((W - Wq) @ X) ** 2)), energy)

    return errors, time.perf_counter() - t0


## 6 · Run the methods

Run each method for every configured dataset. The following cells display comparisons.


In [8]:
CASES = load_cases(W_PATH, X_CAL_PATH, DATASET_NAME)
RESULTS = {}

for label, case in CASES.items():
    W, X = case["W"], case["X"]
    sel = select_format(W, X)
    base_err, base_s = full_baseline_eval(W, X)
    opt_err, bf_s = brute_force_reference(W, X)
    p = sel["plan"]
    scores = sel["scores"].set_index("Format")
    RESULTS[label] = dict(
        winner = sel["winner"],
        topk_err = {f: float(scores.loc[f, "Selected_relative_error_pct"]) for f in scores.index},
        base_err = base_err, opt_err = opt_err,
        retained_pct = 100 * p["terms"] / p["total"],
        topk_s = sel["selection_seconds"], base_s = base_s, bf_s = bf_s)
    print(f"[{label}] done  |  top-k pick: {sel['winner']:<8} |  top-k {sel['selection_seconds']:.3f}s"
          f"  full-baseline {base_s:.3f}s  brute-force {bf_s:.3f}s")


[C] done  |  top-k pick: E2M1     |  top-k 0.231s  full-baseline 0.206s  brute-force 2.100s


## 7 · Comparison results

In [9]:
_GOOD = "background-color:#d4edda;color:#155724;font-weight:bold"
_BAD = "background-color:#f8d7da;color:#721c24;font-weight:bold"

def _best_in_row(tol = 1e-9):

    def f(row):
        vals = pd.to_numeric(row, errors = "coerce")
        m = vals.min()
        return [_GOOD if (pd.notna(v) and abs(v - m) <= tol * max(1.0, abs(m))) else "" for v in vals]

    return f

def _flag(v):
    if isinstance(v, str) and v.startswith("✅"):
        return _GOOD

    if isinstance(v, str) and v.startswith("❌"):
        return _BAD

    return ""

def show_matrix(df, title, fmt = None, best = None, flag_cols = None):
    """Styled matrix. best='min' highlights the row-wise best (smallest) cell(s) in green;
    flag_cols colour the check-mark columns. Falls back to a plain table if styling is unavailable."""

    try:
        sty = df.style.set_caption(title)
        if fmt is not None:
            sty = sty.format(fmt, na_rep = "-")

        sty = sty.set_table_styles([
            {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold"),
                                              ("text-align", "left"), ("padding", "6px 0")]},
            {"selector": "th", "props": [("text-align", "center"), ("background-color", "#eef1f6"),
                                         ("color", "#111")]},
            {"selector": "td", "props": [("text-align", "center")]},
        ])
        if best == "min":
            sty = sty.apply(_best_in_row(), axis = 1)

        if flag_cols:
            mapper = getattr(sty, "map", None) or sty.applymap
            sty = mapper(_flag, subset = flag_cols)

        display(sty)
    except Exception:
        print(title)
        display(df)


In [10]:
labels = list(RESULTS)
fmts = [f for f in SCORED_FORMATS if all(f in RESULTS[l]["topk_err"] for l in labels)]
idx = pd.Index(labels, name = "Dataset")

def to_matrix(key):
    m = pd.DataFrame({f: [RESULTS[l][key][f] for l in labels] for f in fmts}, index = idx)
    m.columns.name = "Format"
    return m

def to_rank(m):
    return m.round(9).rank(axis = 1, method = "min").astype(int)

def best_set(row, tol = 1e-9):
    m = row.min()
    return [f for f in row.index if abs(row[f] - m) <= tol * max(1.0, abs(m))]

topk_err, base_err, opt_err = to_matrix("topk_err"), to_matrix("base_err"), to_matrix("opt_err")
topk_rank, base_rank, opt_rank = to_rank(topk_err), to_rank(base_err), to_rank(opt_err)


### 7.1 · Winners

Show each method’s best format. A check mark means the top-k choice belongs to the reference’s best set, using the configured tie tolerance.


In [11]:
rows = []

for l in labels:
    chosen = RESULTS[l]["winner"]
    b_best, o_best = best_set(base_err.loc[l]), best_set(opt_err.loc[l])
    rows.append({
        "Top-k winner": chosen,
        "Full-baseline best": " = ".join(b_best),
        "Brute-force best": " = ".join(o_best),
        "Top-k in full-baseline best": "✅" if chosen in b_best else "❌",
        "Top-k in brute-force best": "✅" if chosen in o_best else "❌",
        "Regret vs full baseline (pp)": base_err.loc[l, chosen] - base_err.loc[l].min(),
        "Regret vs brute force (pp)": opt_err.loc[l, chosen] - opt_err.loc[l].min(),
    })

winner_table = pd.DataFrame(rows, index = idx)
show_matrix(winner_table, "Winner per dataset and selection regret (pp = percentage points of output error)",
            fmt = {"Regret vs full baseline (pp)": "{:.4f}", "Regret vs brute force (pp)": "{:.4f}"},
            flag_cols = ["Top-k in full-baseline best", "Top-k in brute-force best"])


,Top-k winner,Full-baseline best,Brute-force best,Top-k in full-baseline best,Top-k in brute-force best,Regret vs full baseline (pp),Regret vs brute force (pp)
Dataset,,,,,,,
C,E2M1,E2M1,E2M1,✅,✅,0.0000,0.0000


### 7.2 · Rankings

Rank 1 is best; ties share a rank. Compare rows to see whether top-k preserves the full-baseline ranking and whether exponent optimization changes it.


In [12]:
show_matrix(topk_rank, "Rank - Top-k selector (selected-term error)", fmt = "{:d}", best = "min")
show_matrix(base_rank, "Rank - Full baseline (actual output error, baseline exponents)", fmt = "{:d}", best = "min")
show_matrix(opt_rank, "Rank - Brute-force reference (actual output error, optimised exponents)", fmt = "{:d}", best = "min")


Format,E2M1,E1M2,MXUINT4,MXINT4,BFP4
Dataset,,,,,
C,1,4,5,2,2


Format,E2M1,E1M2,MXUINT4,MXINT4,BFP4
Dataset,,,,,
C,1,4,5,2,2


Format,E2M1,E1M2,MXUINT4,MXINT4,BFP4
Dataset,,,,,
C,1,5,2,2,2


### 7.3 · Relative errors

Lower is better; green marks the best value. Top-k uses a selected-term reference, so compare its ranking with the other methods rather than comparing the percentages directly.


In [13]:
show_matrix(topk_err, "Relative error (%) - Top-k selector (selected terms)", fmt = "{:.4f}", best = "min")
show_matrix(base_err, "Relative output error (%) - Full baseline", fmt = "{:.4f}", best = "min")
show_matrix(opt_err, "Relative output error (%) - Brute-force reference", fmt = "{:.4f}", best = "min")
show_matrix(base_err - opt_err, "Error reduction from exponent optimisation (pp) = Full baseline - Brute force",
            fmt = "{:.4f}")


Format,E2M1,E1M2,MXUINT4,MXINT4,BFP4
Dataset,,,,,
C,12.2480,14.1829,15.3292,14.1662,14.1662


Format,E2M1,E1M2,MXUINT4,MXINT4,BFP4
Dataset,,,,,
C,12.5523,15.8172,17.1262,15.8080,15.8080


Format,E2M1,E1M2,MXUINT4,MXINT4,BFP4
Dataset,,,,,
C,12.0307,14.0042,13.4521,13.4521,13.4521


Format,E2M1,E1M2,MXUINT4,MXINT4,BFP4
Dataset,,,,,
C,0.5217,1.8130,3.6742,2.3559,2.3559


### 7.4 · Ranking agreement

Kendall tau-b measures agreement: 1 means the same order and −1 means the reverse. Exact rank match means every rank agrees. Brute-force comparisons also include the effect of optimizing exponents.


In [14]:
def tau(a, b):
    if a.nunique() < 2 or b.nunique() < 2:
        return float("nan")

    return float(kendalltau(a.values, b.values, variant = "b").statistic)

agree = pd.DataFrame({
    "tau: Top-k vs Full baseline": [tau(topk_rank.loc[l], base_rank.loc[l]) for l in labels],
    "tau: Top-k vs Brute force": [tau(topk_rank.loc[l], opt_rank.loc[l]) for l in labels],
    "tau: Full baseline vs Brute force": [tau(base_rank.loc[l], opt_rank.loc[l]) for l in labels],
    "Exact rank match: Top-k = Full baseline": ["✅" if (topk_rank.loc[l] == base_rank.loc[l]).all() else "❌" for l in labels],
    "Exact rank match: Top-k = Brute force": ["✅" if (topk_rank.loc[l] == opt_rank.loc[l]).all() else "❌" for l in labels],
}, index = idx)
show_matrix(agree, "Ranking agreement between methods",
            fmt = {c: "{:.3f}" for c in agree.columns if c.startswith("tau")},
            flag_cols = [c for c in agree.columns if c.startswith("Exact")])


,tau: Top-k vs Full baseline,tau: Top-k vs Brute force,tau: Full baseline vs Brute force,Exact rank match: Top-k = Full baseline,Exact rank match: Top-k = Brute force
Dataset,,,,,
C,1.000,0.630,0.630,✅,❌


### 7.5 · Work and runtime

Retained terms show the fraction of multiplication terms selected. Top-k still quantizes every candidate matrix. Sparse operations add overhead, so fewer terms may not mean a faster run. Times are single-run measurements; a speed-up below 1 means top-k was slower.


In [15]:
cost = pd.DataFrame({
    "Retained terms (%)": [RESULTS[l]["retained_pct"] for l in labels],
    "Skipped terms (%)": [100 - RESULTS[l]["retained_pct"] for l in labels],
    "Top-k time (s)": [RESULTS[l]["topk_s"] for l in labels],
    "Full baseline time (s)": [RESULTS[l]["base_s"] for l in labels],
    "Brute-force time (s)": [RESULTS[l]["bf_s"] for l in labels],
    "Speed-up vs Full baseline (x)": [RESULTS[l]["base_s"] / RESULTS[l]["topk_s"] for l in labels],
    "Speed-up vs Brute force (x)": [RESULTS[l]["bf_s"] / RESULTS[l]["topk_s"] for l in labels],
}, index = idx)
show_matrix(cost, "Cost comparison",
            fmt = {"Retained terms (%)": "{:.2f}", "Skipped terms (%)": "{:.2f}", "Top-k time (s)": "{:.4f}",
                 "Full baseline time (s)": "{:.4f}", "Brute-force time (s)": "{:.4f}",
                 "Speed-up vs Full baseline (x)": "{:.2f}", "Speed-up vs Brute force (x)": "{:.2f}"})


,Retained terms (%),Skipped terms (%),Top-k time (s),Full baseline time (s),Brute-force time (s),Speed-up vs Full baseline (x),Speed-up vs Brute force (x)
Dataset,,,,,,,
C,36.00,64.00,0.2314,0.2060,2.0996,0.89,9.07


### 7.6 Verdict (computed from the tables above)

In [16]:
n = len(labels)
hit_b = int((winner_table["Top-k in full-baseline best"] == "✅").sum())
hit_o = int((winner_table["Top-k in brute-force best"] == "✅").sum())
rank_b = int((agree["Exact rank match: Top-k = Full baseline"] == "✅").sum())
print(f"Datasets compared                                   : {n}  ({', '.join(labels)})")
print(f"Top-k pick is optimal vs full baseline              : {hit_b}/{n}   (max regret {winner_table['Regret vs full baseline (pp)'].max():.4f} pp)")
print(f"Top-k pick is optimal vs brute-force reference      : {hit_o}/{n}   (max regret {winner_table['Regret vs brute force (pp)'].max():.4f} pp)")
print(f"Identical full ranking, top-k vs full baseline      : {rank_b}/{n}")
print(f"Mean retained multiplication terms                  : {cost['Retained terms (%)'].mean():.2f} %")
print(f"Mean speed-up vs full baseline / brute force        : {cost['Speed-up vs Full baseline (x)'].mean():.2f}x / {cost['Speed-up vs Brute force (x)'].mean():.2f}x")


Datasets compared                                   : 1  (C)
Top-k pick is optimal vs full baseline              : 1/1   (max regret 0.0000 pp)
Top-k pick is optimal vs brute-force reference      : 1/1   (max regret 0.0000 pp)
Identical full ranking, top-k vs full baseline      : 1/1
Mean retained multiplication terms                  : 36.00 %
Mean speed-up vs full baseline / brute force        : 0.89x / 9.07x


## 8 · Export (optional)
Writes every comparison matrix above as CSV.

In [17]:
if EXPORT_RESULTS:
    OUTPUT_DIR.mkdir(parents = True, exist_ok = True)

    for name, df in [("winner_table", winner_table), ("rank_topk", topk_rank), ("rank_full_baseline", base_rank),
                     ("rank_brute_force", opt_rank), ("error_topk_selected_pct", topk_err),
                     ("error_full_baseline_pct", base_err), ("error_brute_force_pct", opt_err),
                     ("agreement", agree), ("cost", cost)]:
        df.to_csv(OUTPUT_DIR / f"{name}.csv")

    print("Comparison matrices saved to", OUTPUT_DIR.resolve())


Comparison matrices saved to C:\Users\Admin\OneDrive\Desktop\System\Algo-2\topk_vs_bruteforce_results


## Keep in mind

A good selected-term score does not guarantee the best full-output result. Top-k still quantizes the full weight matrix. Fewer scored terms may not make the run faster. All errors here use calibration inputs.

Format background: [OCP MX](https://www.opencompute.org/documents/ocp-microscaling-formats-mx-v1-0-spec-final-pdf), [FP4 quantization](https://proceedings.mlr.press/v267/wang25ae.html), [Flexpoint](https://arxiv.org/abs/1711.02213).
